# ☀️ PowerFlow — Solar Generation Forecast Model

> **Goal:** Predict AC Power output of solar panels using weather sensor data and time features.

**Pipeline:**
```
Dataset → Data Cleaning → EDA → Feature Engineering → Train/Test Split
       → XGBoost Model → MAE/RMSE/R² → Actual vs Predicted → Save (.pkl) → Predict()
```

**Dataset:** Kaggle Solar Power Generation — Plant 1  
**Target Variable:** `AC_POWER` (kW output from inverter)

## 📦 0. Install & Import Libraries

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['xgboost', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'joblib']:
    install(pkg)

print('All packages ready!')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import joblib, os

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.titlesize': 13, 'axes.titleweight': 'bold'})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Libraries imported!')

---
## 📂 1. Load Dataset

In [ ]:
BASE_PATH    = r'data/kaggle/solar-power-generation'
GEN_PATH     = os.path.join(BASE_PATH, 'Plant_1_Generation_Data.csv')
WEATHER_PATH = os.path.join(BASE_PATH, 'Plant_1_Weather_Sensor_Data.csv')
MODEL_PATH   = 'models/solar_ac_power_xgb.pkl'
os.makedirs('models', exist_ok=True)

gen_df     = pd.read_csv(GEN_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

print('Generation Data :', gen_df.shape[0], 'rows x', gen_df.shape[1], 'cols')
print('Weather Data    :', weather_df.shape[0], 'rows x', weather_df.shape[1], 'cols')
print()
print('Generation columns:', gen_df.columns.tolist())
print('Weather columns   :', weather_df.columns.tolist())

In [ ]:
gen_df.head()

In [ ]:
weather_df.head()

---
## 🧹 2. Data Cleaning

In [ ]:
# 2.1  Parse Timestamps
# Generation uses 'dd-mm-yyyy HH:MM', Weather uses 'yyyy-mm-dd HH:MM:SS'
gen_df['DATE_TIME']     = pd.to_datetime(gen_df['DATE_TIME'],     dayfirst=True)
weather_df['DATE_TIME'] = pd.to_datetime(weather_df['DATE_TIME'])

print('Gen date range   :', gen_df['DATE_TIME'].min(), '->', gen_df['DATE_TIME'].max())
print('Weather date range:', weather_df['DATE_TIME'].min(), '->', weather_df['DATE_TIME'].max())

In [ ]:
# 2.2  Aggregate generation per timestamp (22 inverters -> plant-level sum)
gen_agg = gen_df.groupby('DATE_TIME').agg(
    DC_POWER    = ('DC_POWER',    'sum'),
    AC_POWER    = ('AC_POWER',    'sum'),
    DAILY_YIELD = ('DAILY_YIELD', 'mean'),
    TOTAL_YIELD = ('TOTAL_YIELD', 'sum')
).reset_index()

print('Aggregated Generation shape:', gen_agg.shape)
gen_agg.head()

In [ ]:
# 2.3  Merge Generation + Weather on DATE_TIME
df = pd.merge(
    gen_agg,
    weather_df[['DATE_TIME', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']],
    on='DATE_TIME', how='inner'
)
print('Merged shape:', df.shape)
df.head()

In [ ]:
# 2.4  Null value check
null_df = df.isnull().sum().reset_index()
null_df.columns = ['Column', 'Null Count']
null_df['Null %'] = (null_df['Null Count'] / len(df) * 100).round(2)
print(null_df.to_string(index=False))

In [ ]:
# 2.5  Drop duplicates and sort by time
before = len(df)
df.drop_duplicates(subset='DATE_TIME', inplace=True)
print('Duplicates removed:', before - len(df))

df.sort_values('DATE_TIME', inplace=True)
df.reset_index(drop=True, inplace=True)

# 2.6  Remove negative power values
neg_mask = (df['AC_POWER'] < 0) | (df['DC_POWER'] < 0)
print('Negative power rows removed:', neg_mask.sum())
df = df[~neg_mask].reset_index(drop=True)

# 2.7  Cap extreme outliers at 99.5th percentile
for col in ['AC_POWER', 'DC_POWER', 'IRRADIATION']:
    cap = df[col].quantile(0.995)
    clipped = (df[col] > cap).sum()
    df[col] = df[col].clip(upper=cap)
    print(f'  {col}: capped {clipped} values at {cap:.2f}')

print()
print('Clean dataset shape:', df.shape)

---
## 📊 3. Exploratory Data Analysis (EDA)

In [ ]:
# 3.1 Statistical Summary
df.describe().round(2)

In [ ]:
# 3.2 AC Power over time
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(df['DATE_TIME'], df['AC_POWER'], color='#f0a500', alpha=0.8, linewidth=0.8)
ax.fill_between(df['DATE_TIME'], df['AC_POWER'], alpha=0.2, color='#f0a500')
ax.set_title('AC Power Output Over Time — Plant 1')
ax.set_xlabel('Date')
ax.set_ylabel('AC Power (kW)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Feature Distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, clr in zip(axes,
                          ['AC_POWER', 'DC_POWER', 'IRRADIATION'],
                          ['#e74c3c', '#3498db', '#f39c12']):
    ax.hist(df[col], bins=50, color=clr, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(), color='white', linestyle='--', linewidth=1.5, label='Mean')
    ax.set_title(col)
    ax.legend()
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 Correlation Heatmap
num_cols = ['AC_POWER', 'DC_POWER', 'DAILY_YIELD', 'IRRADIATION',
             'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={'size': 10})
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# 3.5 Irradiation vs AC Power (scatter)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(df['IRRADIATION'], df['AC_POWER'],
                     alpha=0.3, s=8, c=df['MODULE_TEMPERATURE'], cmap='plasma')
plt.colorbar(sc, ax=axes[0], label='Module Temp (C)')
axes[0].set_title('Irradiation vs AC Power')
axes[0].set_xlabel('Irradiation (W/m2)')
axes[0].set_ylabel('AC Power (kW)')

axes[1].scatter(df['AMBIENT_TEMPERATURE'], df['AC_POWER'],
                alpha=0.3, s=8, color='#16a085')
axes[1].set_title('Ambient Temp vs AC Power')
axes[1].set_xlabel('Ambient Temp (C)')
axes[1].set_ylabel('AC Power (kW)')

plt.suptitle('Key Feature Relationships', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.6 Average AC Power by Hour of Day
df['_hour'] = df['DATE_TIME'].dt.hour
hourly = df.groupby('_hour')['AC_POWER'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hourly.index, hourly.values,
       color=plt.cm.YlOrRd(hourly.values / hourly.values.max()),
       edgecolor='white', linewidth=0.5)
ax.set_title('Average AC Power by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Mean AC Power (kW)')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

df.drop(columns=['_hour'], inplace=True)

---
## ⚙️ 4. Feature Engineering

In [ ]:
# 4.1 Time Features
df['hour']                  = df['DATE_TIME'].dt.hour
df['minute']                = df['DATE_TIME'].dt.minute
df['day_of_week']           = df['DATE_TIME'].dt.dayofweek      # 0=Monday
df['day_of_month']          = df['DATE_TIME'].dt.day
df['month']                 = df['DATE_TIME'].dt.month
df['week']                  = df['DATE_TIME'].dt.isocalendar().week.astype(int)

# Cyclic encoding: captures periodicity (hour 23 is close to hour 0)
df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df['minutes_since_midnight'] = df['hour'] * 60 + df['minute']
df['is_daytime']             = ((df['hour'] >= 6) & (df['hour'] < 18)).astype(int)

print('Time features added')

In [ ]:
# 4.2 Lag Features
# 15-min intervals: lag_1 = 15 min ago, lag_4 = 1 hr, lag_8 = 2 hr
df['ac_lag_1']  = df['AC_POWER'].shift(1)
df['ac_lag_4']  = df['AC_POWER'].shift(4)
df['ac_lag_8']  = df['AC_POWER'].shift(8)
df['dc_lag_1']  = df['DC_POWER'].shift(1)
df['irr_lag_1'] = df['IRRADIATION'].shift(1)

# 4.3 Rolling Window Features
df['ac_roll_mean_4']  = df['AC_POWER'].rolling(window=4).mean()   # 1-hr rolling avg
df['ac_roll_mean_8']  = df['AC_POWER'].rolling(window=8).mean()   # 2-hr rolling avg
df['irr_roll_mean_4'] = df['IRRADIATION'].rolling(window=4).mean()
df['ac_roll_std_4']   = df['AC_POWER'].rolling(window=4).std()    # volatility

# 4.4 Derived Physical Features
df['temp_delta'] = df['MODULE_TEMPERATURE'] - df['AMBIENT_TEMPERATURE']

# Drop NaN rows created by lag/rolling
before = len(df)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print('NaN rows dropped:', before - len(df))
print('Final shape     :', df.shape)
print('All features    :', df.columns.tolist())

---
## ✂️ 5. Train / Test Split

In [ ]:
TARGET = 'AC_POWER'

# Exclude target, datetime, leaky columns
DROP_COLS    = ['DATE_TIME', 'AC_POWER', 'DC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD']
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]

print('Features (' + str(len(FEATURE_COLS)) + '):')
for f in FEATURE_COLS:
    print('  -', f)

In [ ]:
# Time-based split: 80% train / 20% test
# (Random split would cause data leakage in time-series!)
split_idx = int(len(df) * 0.80)

train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET]
X_test,  y_test  = test_df[FEATURE_COLS],  test_df[TARGET]

print('Train:', len(X_train), 'samples  (',
      train_df['DATE_TIME'].min().date(), '->', train_df['DATE_TIME'].max().date(), ')')
print('Test :', len(X_test),  'samples  (',
      test_df['DATE_TIME'].min().date(),  '->', test_df['DATE_TIME'].max().date(),  ')')

In [ ]:
# Visualise the train/test split
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(train_df['DATE_TIME'], y_train, color='#2ecc71', alpha=0.8, linewidth=0.8, label='Train')
ax.plot(test_df['DATE_TIME'],  y_test,  color='#e74c3c', alpha=0.8, linewidth=0.8, label='Test')
ax.axvline(df['DATE_TIME'].iloc[split_idx], color='white',
           linestyle='--', linewidth=1.5, label='Split point')
ax.set_title('Train / Test Split — AC Power')
ax.set_xlabel('Date')
ax.set_ylabel('AC Power (kW)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 🤖 6. XGBoost Model Training

In [ ]:
model = XGBRegressor(
    n_estimators          = 500,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 3,
    reg_alpha             = 0.1,    # L1 regularisation
    reg_lambda            = 1.0,    # L2 regularisation
    random_state          = RANDOM_STATE,
    n_jobs                = -1,
    tree_method           = 'hist', # fast histogram-based training
    early_stopping_rounds = 30,
    eval_metric           = 'rmse'
)

print('Training XGBoost model...')
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50
)
print('Training complete!')
print('Best iteration:', model.best_iteration)

---
## 📈 7. Model Evaluation — MAE / RMSE / R²

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    print('-- ' + label + ' --')
    print(f'  MAE  : {mae:>10.2f} kW')
    print(f'  RMSE : {rmse:>10.2f} kW')
    print(f'  R2   : {r2:>10.4f}')
    print(f'  MAPE : {mape:>10.2f} %')
    print()
    return mae, rmse, r2, mape

print('='*40)
tr = metrics(y_train, y_pred_train, 'TRAIN')
te = metrics(y_test,  y_pred_test,  'TEST')
print('='*40)

In [ ]:
# Metrics Bar Chart
labels   = ['MAE (kW)', 'RMSE (kW)', 'R2', 'MAPE (%)']
train_v  = [tr[0], tr[1], tr[2], tr[3]]
test_v   = [te[0], te[1], te[2], te[3]]
clrs     = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, label, tv, ev, clr in zip(axes, labels, train_v, test_v, clrs):
    ax.bar(['Train', 'Test'], [tv, ev], color=[clr, clr],
           alpha=[0.5, 1.0], edgecolor='white')
    ax.set_title(label)
    for i, val in enumerate([tv, ev]):
        ax.text(i, val * 1.01, f'{val:.3f}', ha='center', va='bottom',
                fontsize=10, fontweight='bold')
plt.suptitle('Model Performance — Train vs Test', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# XGBoost Learning Curve
results = model.evals_result()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(results['validation_0']['rmse'], label='Train RMSE', color='#2ecc71', linewidth=1.5)
ax.plot(results['validation_1']['rmse'], label='Test RMSE',  color='#e74c3c', linewidth=1.5)
ax.axvline(model.best_iteration, color='white', linestyle='--',
           linewidth=1.2, label='Best iter: ' + str(model.best_iteration))
ax.set_title('XGBoost Learning Curve')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('RMSE (kW)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS)
importances_sorted = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors_fi = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(importances_sorted)))
importances_sorted.plot(kind='barh', ax=ax, color=colors_fi, edgecolor='none')
ax.set_title('XGBoost Feature Importance (gain)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
## 🎯 8. Actual vs Predicted

In [ ]:
# Full test set time-series overlay
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(test_df['DATE_TIME'].values, y_test.values,
        label='Actual',    color='#3498db', alpha=0.9, linewidth=1.2)
ax.plot(test_df['DATE_TIME'].values, y_pred_test,
        label='Predicted', color='#e74c3c', alpha=0.75, linewidth=1.0, linestyle='--')
ax.set_title('Actual vs Predicted — AC Power (Test Set)')
ax.set_xlabel('Date')
ax.set_ylabel('AC Power (kW)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Zoomed: first 3 days of test set (288 = 3 days x 96 intervals per day)
zoom      = test_df.iloc[:288].copy()
zoom_pred = y_pred_test[:288]

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(zoom['DATE_TIME'].values, zoom['AC_POWER'].values,
        label='Actual', color='#3498db', linewidth=1.5)
ax.plot(zoom['DATE_TIME'].values, zoom_pred,
        label='Predicted', color='#e74c3c', linewidth=1.5, linestyle='--')
ax.fill_between(zoom['DATE_TIME'].values, zoom['AC_POWER'].values, zoom_pred,
                alpha=0.15, color='#e74c3c', label='Error')
ax.set_title('Actual vs Predicted — Zoomed (First 3 Days of Test Set)')
ax.set_xlabel('Date')
ax.set_ylabel('AC Power (kW)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b %H:%M'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Predicted vs Actual (perfect = diagonal line)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test.values, y_pred_test, alpha=0.2, s=6, color='#9b59b6')
lims = [min(y_test.min(), y_pred_test.min()),
        max(y_test.max(), y_pred_test.max())]
ax.plot(lims, lims, 'w--', linewidth=1.5, label='Perfect prediction')
ax.set_title('Predicted vs Actual AC Power')
ax.set_xlabel('Actual AC Power (kW)')
ax.set_ylabel('Predicted AC Power (kW)')
r2_val = r2_score(y_test, y_pred_test)
ax.text(0.05, 0.92, 'R2 = ' + str(round(r2_val, 4)), transform=ax.transAxes,
        fontsize=12, color='white', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residual Analysis
residuals = y_test.values - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(residuals, bins=60, color='#16a085', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='white', linestyle='--', linewidth=1.5)
axes[0].set_title('Residual Distribution')
axes[0].set_xlabel('Residual (kW)')
axes[0].set_ylabel('Count')

axes[1].scatter(y_pred_test, residuals, alpha=0.2, s=6, color='#16a085')
axes[1].axhline(0, color='white', linestyle='--', linewidth=1.5)
axes[1].set_title('Residuals vs Predicted')
axes[1].set_xlabel('Predicted AC Power (kW)')
axes[1].set_ylabel('Residual (kW)')

plt.suptitle('Residual Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 💾 9. Save Model (.pkl)

In [ ]:
model_artifact = {
    'model'        : model,
    'feature_cols' : FEATURE_COLS,
    'target'       : TARGET,
    'train_end'    : str(train_df['DATE_TIME'].max()),
    'test_mae'     : te[0],
    'test_rmse'    : te[1],
    'test_r2'      : te[2],
}

joblib.dump(model_artifact, MODEL_PATH)

size_kb = os.path.getsize(MODEL_PATH) / 1024
print('Model saved to:', MODEL_PATH)
print('File size (KB) :', round(size_kb, 1))
print('Features       :', len(FEATURE_COLS))
print('Test MAE       :', round(te[0], 2), 'kW')
print('Test R2        :', round(te[2], 4))

---
## 🔮 10. Prediction Function

Reusable function — copy-paste anywhere in the PowerFlow project.

In [ ]:
import joblib
import numpy as np
import pandas as pd


def predict_ac_power(
    timestamp: str,
    irradiation: float,
    ambient_temperature: float,
    module_temperature: float,
    ac_lag_1: float       = 0.0,
    ac_lag_4: float       = 0.0,
    ac_lag_8: float       = 0.0,
    dc_lag_1: float       = 0.0,
    irr_lag_1: float      = 0.0,
    ac_roll_mean_4: float = 0.0,
    ac_roll_mean_8: float = 0.0,
    irr_roll_mean_4: float = 0.0,
    ac_roll_std_4: float  = 0.0,
    model_path: str       = 'models/solar_ac_power_xgb.pkl'
) -> dict:
    """
    Predict AC Power output for a solar plant at a given timestamp.

    Parameters
    ----------
    timestamp            : ISO datetime string, e.g. '2020-06-10 14:30:00'
    irradiation          : Solar irradiation in W/m2
    ambient_temperature  : Ambient temperature in degrees C
    module_temperature   : Solar module temperature in degrees C
    ac_lag_1/4/8         : AC power at t-15min / t-1hr / t-2hr (kW)
    dc_lag_1             : DC power at t-15min (kW)
    irr_lag_1            : Irradiation at t-15min
    ac_roll_mean_4/8     : 1-hr / 2-hr rolling mean of AC power
    irr_roll_mean_4      : 1-hr rolling mean of irradiation
    ac_roll_std_4        : 1-hr rolling std of AC power
    model_path           : Path to saved .pkl model artifact

    Returns
    -------
    dict with keys: predicted_ac_power_kw, timestamp, model_path
    """
    artifact = joblib.load(model_path)
    model_   = artifact['model']
    features = artifact['feature_cols']

    dt     = pd.Timestamp(timestamp)
    hour   = dt.hour
    minute = dt.minute
    month  = dt.month

    row = {
        'AMBIENT_TEMPERATURE'    : ambient_temperature,
        'MODULE_TEMPERATURE'     : module_temperature,
        'IRRADIATION'            : irradiation,
        'hour'                   : hour,
        'minute'                 : minute,
        'day_of_week'            : dt.dayofweek,
        'day_of_month'           : dt.day,
        'month'                  : month,
        'week'                   : dt.isocalendar().week,
        'hour_sin'               : np.sin(2 * np.pi * hour  / 24),
        'hour_cos'               : np.cos(2 * np.pi * hour  / 24),
        'month_sin'              : np.sin(2 * np.pi * month / 12),
        'month_cos'              : np.cos(2 * np.pi * month / 12),
        'minutes_since_midnight' : hour * 60 + minute,
        'is_daytime'             : int(6 <= hour < 18),
        'ac_lag_1'               : ac_lag_1,
        'ac_lag_4'               : ac_lag_4,
        'ac_lag_8'               : ac_lag_8,
        'dc_lag_1'               : dc_lag_1,
        'irr_lag_1'              : irr_lag_1,
        'ac_roll_mean_4'         : ac_roll_mean_4,
        'ac_roll_mean_8'         : ac_roll_mean_8,
        'irr_roll_mean_4'        : irr_roll_mean_4,
        'ac_roll_std_4'          : ac_roll_std_4,
        'temp_delta'             : module_temperature - ambient_temperature,
    }

    X    = pd.DataFrame([row])[features]
    pred = float(model_.predict(X)[0])
    pred = max(0.0, pred)  # power cannot be negative

    return {
        'predicted_ac_power_kw' : round(pred, 3),
        'timestamp'             : str(dt),
        'model_path'            : model_path
    }

In [ ]:
# Quick test of the prediction function
result = predict_ac_power(
    timestamp           = '2020-06-10 12:30:00',
    irradiation         = 0.85,
    ambient_temperature = 28.5,
    module_temperature  = 42.3,
    ac_lag_1            = 800.0,
    ac_lag_4            = 750.0,
    ac_lag_8            = 600.0,
    dc_lag_1            = 820.0,
    irr_lag_1           = 0.82,
    ac_roll_mean_4      = 780.0,
    ac_roll_mean_8      = 720.0,
    irr_roll_mean_4     = 0.83,
    ac_roll_std_4       = 30.0,
)

print('Prediction Result')
print('-' * 35)
for k, v in result.items():
    print(f'  {k:<30} : {v}')

---
## ✅ Summary

| Step | Details |
|------|---------|
| **Dataset** | Plant 1 Generation + Weather Sensor (Kaggle Solar Power) |
| **Target** | `AC_POWER` — plant-level aggregated output (kW) |
| **Model** | XGBoost Regressor with early stopping |
| **Features** | 25 features: time cyclics, lags, rolling stats, temp delta |
| **Split** | Time-based 80% train / 20% test (no data leakage) |
| **Saved** | `models/solar_ac_power_xgb.pkl` |

### 🚀 Next Steps
- Add **Plant 2** data for a combined model
- Try **hyperparameter tuning** with `Optuna`
- Integrate with `powerflow_2026_telemetry.csv` for live inference
- Build a **Streamlit dashboard** for real-time predictions